In [ ]:
import sys
import os
# Ensure project root is on path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from src.ml.data_loader import get_engine, load_telemetry, load_failures, load_equipment_ids
from src.ml.feature_prep import (
    label_telemetry, select_features, prepare_xy,
    compute_sample_weights, equipment_train_test_split, temporal_validation_split
)
from src.ml.train_xgb_classifier import setup_mlflow, train_fleet_model, train_fleet_model_with_tuning
from src.ml.evaluate import evaluate_model, evaluate_all_vs_sensor_only, get_feature_importance

In [ ]:
# Pipeline parameters
EQUIPMENT_TYPE = "turbine"  # Options: "turbine", "compressor", "pump"
PREDICTION_HORIZON_HOURS = 72.0
TEST_FRACTION = 0.3

DB_URL = os.environ.get("POSTGRES_URL", "")
print(f"Equipment type: {EQUIPMENT_TYPE}")
print(f"Prediction horizon: {PREDICTION_HORIZON_HOURS}h")
print(f"DB connected: {bool(DB_URL)}")

In [ ]:
engine = get_engine(DB_URL)

telemetry = load_telemetry(engine, EQUIPMENT_TYPE)
failures = load_failures(engine, EQUIPMENT_TYPE)
equipment_ids = load_equipment_ids(engine, EQUIPMENT_TYPE)

print(f"Telemetry rows: {len(telemetry):,}")
print(f"Failure events: {len(failures):,}")
print(f"Equipment units: {len(equipment_ids):,}")
# telemetry.head()

In [ ]:
# Quick look at failure distribution
if len(failures) > 0:
    print("Failure mode distribution:")
    print(failures["failure_mode_code"].value_counts())

In [ ]:
# Label telemetry with failure modes
labeled = label_telemetry(telemetry, failures, PREDICTION_HORIZON_HOURS)

print(f"Label distribution:")
print(labeled["label"].value_counts())

In [ ]:
# Train/test split by equipment ID (prevents data leakage)
train_df, test_df = equipment_train_test_split(labeled, equipment_ids, TEST_FRACTION)

# Temporal validation split within training set
train_df, val_df = temporal_validation_split(train_df, val_fraction=0.15)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

In [ ]:
# Select sensor-only features (what's available in production)
sensor_cols = select_features(train_df, EQUIPMENT_TYPE, mode="sensor_only")
print(f"Sensor features ({len(sensor_cols)}): {sensor_cols}")

X_train, y_train, label_encoder = prepare_xy(train_df, sensor_cols)
X_val, y_val, _ = prepare_xy(val_df, sensor_cols)
X_test, y_test, _ = prepare_xy(test_df, sensor_cols)

print(f"\nClasses: {list(label_encoder.classes_)}")

In [ ]:
# Setup MLflow tracking
setup_mlflow(experiment_name="pdm_failure_classification")

In [ ]:
# Train fleet-level XGBoost classifier (sensor-only features)
sensor_model, sensor_run_id = train_fleet_model(
    X_train, y_train, X_val, y_val,
    label_encoder=label_encoder,
    equipment_type=EQUIPMENT_TYPE,
    mode="sensor_only",
    log_to_mlflow=True
)
print(f"Sensor-only model trained. MLflow run_id: {sensor_run_id}")

In [ ]:
metrics = evaluate_model(
    sensor_model, X_test, y_test, label_encoder,
    dataset_name="test",
    log_to_mlflow=True,
    run_id=sensor_run_id,
    feature_names=sensor_cols
)

print(f"Accuracy:     {metrics['accuracy']:.4f}")
print(f"Macro F1:     {metrics['macro_f1']:.4f}")
print(f"Weighted F1:  {metrics['weighted_f1']:.4f}")
if metrics['roc_auc_macro'] is not None:
    print(f"ROC AUC:      {metrics['roc_auc_macro']:.4f}")

In [ ]:
# Feature importance (also logged as MLflow artifact)
%matplotlib inline
importance = get_feature_importance(sensor_model, sensor_cols, top_n=15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance["feature"], importance["importance"])
ax.set_xlabel("Importance")
ax.set_title(f"{EQUIPMENT_TYPE.title()} - Top Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
# Train full model (includes health/ground-truth columns) for gap analysis
full_cols = select_features(train_df, EQUIPMENT_TYPE, mode="full")
print(f"Full features ({len(full_cols)}): {full_cols}")

X_train_full, y_train_full, _ = prepare_xy(train_df, full_cols)
X_val_full, y_val_full, _ = prepare_xy(val_df, full_cols)
X_test_full, y_test_full, _ = prepare_xy(test_df, full_cols)

full_model, full_run_id = train_fleet_model(
    X_train_full, y_train_full, X_val_full, y_val_full,
    label_encoder=label_encoder,
    equipment_type=EQUIPMENT_TYPE,
    mode="all",
    log_to_mlflow=True
)
print(f"Full model trained. MLflow run_id: {full_run_id}")

In [ ]:
# Compare full vs sensor-only
gap_results = evaluate_all_vs_sensor_only(
    full_model, sensor_model,
    X_test_full, X_test, y_test,
    label_encoder
)

print(f"All model     - Accuracy: {gap_results['all']['accuracy']:.4f}, Macro F1: {gap_results['all']['macro_f1']:.4f}")
print(f"Sensor model  - Accuracy: {gap_results['sensor_only']['accuracy']:.4f}, Macro F1: {gap_results['sensor_only']['macro_f1']:.4f}")
print(f"Accuracy gap: {gap_results['gap']['accuracy_gap']:.4f}")
print(f"Macro F1 gap: {gap_results['gap']['macro_f1_gap']:.4f}")

In [ ]:
# Hyperparameter tuning with cross-validation (sensor-only)
tuned_model, tuned_run_id = train_fleet_model_with_tuning(
    X_train, y_train, X_val, y_val,
    label_encoder=label_encoder,
    equipment_type=EQUIPMENT_TYPE,
    mode="sensor_only",
    n_cv_folds=5,
    n_iter=20,
    log_to_mlflow=True
)

# Evaluate tuned model
tuned_metrics = evaluate_model(
    tuned_model, X_test, y_test, label_encoder,
    dataset_name="test",
    log_to_mlflow=True,
    run_id=tuned_run_id,
    feature_names=sensor_cols
)

print(f"Tuned model - Accuracy: {tuned_metrics['accuracy']:.4f}, Macro F1: {tuned_metrics['macro_f1']:.4f}")
print(f"Baseline    - Accuracy: {metrics['accuracy']:.4f}, Macro F1: {metrics['macro_f1']:.4f}")